In [1]:
# HF access authorization

from hf_auth import init_hf

# Initialize the token securely before loading any models
init_hf()

✅ Hugging Face token successfully loaded!


True

### Example - ViT for image classification

Why resize to 224×224? 

ViT was pretrained at 224; matching size & normalization speeds convergence.

In [4]:
from datasets import load_dataset
from transformers import AutoImageProcessor, ViTForImageClassification, TrainingArguments, Trainer, DefaultDataCollator
from torchvision.transforms import Compose, Resize, RandomHorizontalFlip, ToTensor, Normalize
from PIL import Image
import numpy as np, evaluate, torch

import warnings
warnings.filterwarnings('ignore')

#### What does the "datasets" library do?

1. The command from datasets import load_dataset imports the core function used 
-   To download and prepare datasets from the Hugging Face Hub or local files. 
-   When you run load_dataset("dataset_name"), 
    -   it automatically downloads the data, 
    -   caches it locally, 
    -   and loads it as a memory-efficient Dataset object ready for machine learning models.

#### In addition, datasets library provides a massive suite of tools to clean, transform, and manage your data.

1. .map(): Applies a custom function to every row or batch in the dataset. It supports multi-processing to speed up heavy text tokenization or image resizing.
2. .filter(): Filters out rows based on a specific condition, like removing sentences that are too short.
3. .select() and .shuffle(): Extracts specific rows by their indices or shuffles the dataset randomly for training.
4. .sort(): Sorts the dataset based on the values of one or more columns.
5. train_test_split(): Splits a single dataset into training and testing subsets, similar to Scikit-Learn.
6. .set_format(): Dynamically changes the output format to work natively with PyTorch, TensorFlow, JAX, NumPy, or Pandas without altering the underlying data.

__Look at the documentation for all the other useful things available in dataset__
   

In [6]:
# Data + labels
ds = load_dataset("uoft-cs/cifar10")
id2label = {i: n for i, n in enumerate(ds["train"].features["label"].names)}
label2id = {v: k for k, v in id2label.items()}

README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  120MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 23.9MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [7]:
# Image processor (does resize to 224 + normalization automatically for ViT)
proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

# Preprocess with batched map
def preprocess(examples):
    # Ensure everything is PIL or arrays; processor can handle both
    images = [
        img if isinstance(img, Image.Image) else Image.fromarray(img)
        for img in examples["img"]
    ]
    out = proc(images=images)  # don't set return_tensors; datasets will tensorize later
    return {"pixel_values": out["pixel_values"], "labels": examples["label"]}

train_ds = ds["train"].map(preprocess, batched=True, remove_columns=ds["train"].column_names)
test_ds  = ds["test"].map(preprocess,  batched=True, remove_columns=ds["test"].column_names)

# Convert to torch tensors for Trainer
train_ds.set_format(type="torch", columns=["pixel_values", "labels"])
test_ds.set_format(type="torch", columns=["pixel_values", "labels"])


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [8]:
# Model (pretrained ViT, new 10-class head)
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224-in21k",
    num_labels=10, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True
)

# Baseline: freeze the whole ViT encoder, train only the classification head
for p in model.vit.parameters():
    p.requires_grad = False

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# Compute metrics
metric = evaluate.load("accuracy")
def compute_metrics(p):  # tiny helper
    preds = np.argmax(p.predictions, axis=1)
    return metric.compute(predictions=preds, references=p.label_ids)

In [11]:
# Training arguments inputs
args = TrainingArguments(
    output_dir="vit-cifar10",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=1,
    learning_rate=5e-5,
    weight_decay=0.05,
    warmup_steps=500,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    remove_unused_columns=False,              # If there are any()
    fp16=torch.cuda.is_available(),
)

In [12]:
# TRainer class instantiation and train
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=DefaultDataCollator(),
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())

Epoch,Training Loss,Validation Loss,Accuracy
1,1.639027,1.639506,0.933100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vit.layers.0.attention.q_proj.weight', 'vit.layers.0.attention.q_proj.bias', 'vit.layers.0.attention.k_proj.weight', 'vit.layers.0.attention.k_proj.bias', 'vit.layers.0.attention.v_proj.weight', 'vit.layers.0.attention.v_proj.bias', 'vit.layers.0.attention.o_proj.weight', 'vit.layers.0.attention.o_proj.bias', 'vit.layers.0.layernorm_before.weight', 'vit.layers.0.layernorm_before.bias', 'vit.layers.0.layernorm_after.weight', 'vit.layers.0.layernorm_after.bias', 'vit.layers.0.mlp.fc1.weight', 'vit.layers.0.mlp.fc1.bias', 'vit.layers.0.mlp.fc2.weight', 'vit.layers.0.mlp.fc2.bias', 'vit.layers.1.attention.q_proj.weight', 'vit.layers.1.attention.q_proj.bias', 'vit.layers.1.attention.k_proj.weight', 'vit.layers.1.attention.k_proj.bias', 'vit.layers.1.attention.v_proj.weight', 'vit.layers.1.attention.v_proj.bias', 'vit.layers.1.attention.o_proj.weight', 'vit.layers.1.attention.o_proj.bias', 'vit.layers.1.layernorm_before

Training Loss,Validation Loss,Epoch,Accuracy
1.639027,1.639506,1,0.933100


{'eval_loss': 1.6395056247711182, 'eval_accuracy': 0.9331}


In [ ]:
# One-image inference demo
ex = test_ds[0]  # already tensors
with torch.no_grad():
    out = model(ex["pixel_values"].unsqueeze(0).to(model.device))
    print("Pred:", id2label[out.logits.argmax(-1).item()], "| True:", id2label[ex["labels"].item()])

# 7) Save
trainer.save_model("vit-cifar10/best")

In [ ]:
# If you initialized the image processr as part of training it would get saved else
# save it seapartely

trainer.save_model("vit-cifar10/best")
proc.save_pretrained("vit-cifar10/best")  # writes preprocessor_config.json

In [ ]:
# Predict function
def predict(image_path, model_dir="vit-cifar10/best"):
    # 1) Load model + processor
    processor = AutoImageProcessor.from_pretrained(model_dir)
    model = ViTForImageClassification.from_pretrained(model_dir)
    model.eval()

    # 2) Load + preprocess image
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt")

    # 3) Forward pass
    with torch.no_grad():
        outputs = model(**inputs)
        pred_id = outputs.logits.argmax(-1).item()

    # 4) Map id → label
    return model.config.id2label[pred_id]

# Call function
print(predict("plane.jpg"))
